# 04 - QSVC (Quantum Support Vector Classifier)

Addestra e valuta il modello QSVC, basato sul kernel quantistico a fedeltà (`FidelityQuantumKernel`), sui tre dataset in simulazione ideale (`StatevectorSampler`), secondo la configurazione riportata in `config.DATASET_CONFIGS` (sezione "Configurazione sperimentale" del Capitolo 7).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from config import DATASET_CONFIGS, RANDOM_STATE, QSVC_IDEAL_MAX_CIRCUITS_PER_JOB
from src.pipeline import train_and_evaluate_qsvc
from src.evaluation.plots import plot_confusion_matrix

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
TABLES_DIR = Path.cwd().parent / "results" / "tables"
FIGURES_DIR = Path.cwd().parent / "results" / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
rows = []

for name, cfg in DATASET_CONFIGS.items():
    data = np.load(PROCESSED_DIR / f"{name}.npz")
    result = train_and_evaluate_qsvc(
        cfg["n_components"], cfg["feature_map_reps"], cfg["entanglement"],
        data["X_train"], data["y_train"], data["X_test"], data["y_test"],
        random_state=RANDOM_STATE,
        max_circuits_per_job=QSVC_IDEAL_MAX_CIRCUITS_PER_JOB,
    )
    result["dataset"] = name

    plot_confusion_matrix(
        result["confusion_matrix"], f"QSVC - {name} (simulazione ideale)",
        FIGURES_DIR / f"confusion_matrix_qsvc_{name}_ideale.png", cmap="Blues",
    )

    rows.append(result)

qsvc_results = pd.DataFrame(rows)
qsvc_results.to_csv(TABLES_DIR / "qsvc_ideal_results.csv", index=False)
qsvc_results[["dataset", "accuracy", "precision", "recall", "f1_score",
              "fit_time_s", "n_circuits", "circuit_depth"]]